In [ ]:
from trainer import BaseTrainModule, Trainer
from model_backbone_2 import JSWRegression
from dataset_main import Data_Loader

import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
import matplotlib.pyplot as plt
from torch import optim
import torch.nn as nn
import os

ROOT_PATH = os.path.abspath(os.path.join(os.path.join(os.path.join(os.getcwd(), os.pardir), os.pardir), os.pardir))

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

class MyTrainModule(BaseTrainModule):
    def __init__(self, flag, reduce):
        super().__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        
        self.model = JSWRegression(in_channels=1, out_channels=1)
        self.model = nn.DataParallel(self.model)
        self.model.to(device=self.device)

        pretrain_path = ROOT_PATH + '/experiments/Exp_downstream/JSW_evaluation/parameter/best_jsw_model_{}_{}_vit.pth'.format(flag, reduce)
        state_dict = torch.load(pretrain_path)
        self.model.module.load_state_dict(state_dict)

        if flag == 1:
            self.save_path = ROOT_PATH + '/experiments/Exp_downstream/JSW_evaluation/results/JSW_summary_{}_pre_{}_vit.txt'.format('wo', reduce)
        else:
            self.save_path = ROOT_PATH + '/experiments/Exp_downstream/JSW_evaluation/results/JSW_summary_{}_pre_{}_vit.txt'.format('w', reduce)

        with open(self.save_path, 'a+', encoding='utf-8') as file:
            file.truncate(0)

    def configure_lossfunctions(self):
        self.criterion = nn.MSELoss()

    def configure_optimizers(self, lr):
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr, betas=(0.5, 0.999))

        return self.optimizer

    def configure_scheduler(self, optimizers):
        optimizer = optimizers
        self.scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=100, gamma=0.5)

        return (self.scheduler)

    def configure_logs(self):
        # learning rate
        self.set_log(name='Lr', obj=self.optimizer, category='lr')
        # loss function
        self.set_log(name='Loss_jsw', obj=self.criterion, category='loss', mode='train')
        self.set_log(name='Loss_jsw_val', obj=self.criterion, category='loss', mode='valid')

    def validation_step(self, batch_idx, batch):
        self.model.eval()
        image, jsw = batch

        pre_jsw = self.model(image)
        
        loss_val = torch.sqrt(self.criterion(pre_jsw, jsw))
        pre_jsw = pre_jsw.cpu().detach().numpy()[0][0]
        gt_jsw = jsw.cpu().detach().numpy()[0][0]
                    
        print('Pre: {}, GT: {}'.format(pre_jsw, gt_jsw))

        # result_dict = {'MSE': mean_squared_error(pre_jsw, gt_jsw),
        #                'MAE': mean_absolute_error(pre_jsw, gt_jsw),
        #                'R2': r2_score(pre_jsw, gt_jsw)}
        # print(result_dict)

        f_test = open(self.save_path, 'a')
        f_test.writelines('{},{}\n'.format(pre_jsw, gt_jsw))

        image_list = [image]

        return loss_val, image_list

    def show_single_log_image(self, image_box):
        image = image_box
        plt.imshow(image[0][0])
        plt.show()


if __name__ == "__main__":
    # Data Loading
    image_size = 224
    transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(0, 1)
                                    ])

    train_dataset = Data_Loader('main_JSW_train_data.json', transform)
    valid_dataset = Data_Loader('main_JSW_test_data.json', transform)

    for reduce in [100]:
        mytrainmodule = MyTrainModule(flag=1, reduce=reduce)
        trainer = Trainer(train_module=mytrainmodule, train_dataset=train_dataset, valid_dataset=valid_dataset)
        trainer.configure(batch_size=1, epochs=1, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                        result_number=1, lr=1e-5)
        trainer.fit_valid()
